# 야간 운전 시각 개선 시스템 — Colab 학습 노트북

**모델**: Zero-DCE + Retinex 기반 손실함수  
**학습 전략**: 2단계 (LOL 사전학습 → 커스텀 데이터 파인튜닝)

## 실행 순서
1. 환경 설치
2. Google Drive 마운트
3. 저장소 코드 로드
4. 데이터 준비 (LOL 다운로드 + 커스텀 데이터 복사)
5. Stage 1: LOL 사전학습 *(STEP 5 이후)*
6. Stage 2: 커스텀 파인튜닝 *(STEP 5 이후)*
7. 결과 시각화
8. 모델 Drive에 저장

---
## 1. 환경 설치

In [ ]:
# GPU 확인
import torch
print('CUDA 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# 필요 패키지 설치
%pip install -q gdown scikit-image tqdm onnx onnxruntime

---
## 2. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Drive 내 프로젝트 루트 경로 설정 (본인 경로에 맞게 수정)
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/night-vision'
DRIVE_CUSTOM_DATA  = '/content/drive/MyDrive/night-vision-data'  # 커스텀 데이터 위치
DRIVE_MODEL_SAVE   = '/content/drive/MyDrive/night-vision-models'

import os
os.makedirs(DRIVE_MODEL_SAVE, exist_ok=True)
print('Drive 마운트 완료.')

---
## 3. 저장소 코드 로드

**방법 A** (권장): GitHub에서 클론  
**방법 B**: Drive에 올린 코드를 Colab으로 복사

In [ ]:
import os

REPO_URL = 'https://github.com/mia2583/night-vision.git'  # 실제 URL로 변경
PROJECT_DIR = '/content/night-vision'

# ── 방법 A: GitHub 클론 ──────────────────────────
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull

# ── 방법 B: Drive에서 복사 (A 실패 시 주석 해제) ──
# import shutil
# shutil.copytree(DRIVE_PROJECT_ROOT, PROJECT_DIR, dirs_exist_ok=True)

os.chdir(PROJECT_DIR)
print('작업 디렉터리:', os.getcwd())

---
## 4. 데이터 준비

In [ ]:
# 경로 설정
LOL_DIR    = 'data/lol'
CUSTOM_DIR = 'data/custom'
DATA_DIR   = 'data/processed'

os.makedirs(LOL_DIR,    exist_ok=True)
os.makedirs(CUSTOM_DIR, exist_ok=True)

In [ ]:
# LOL Dataset 다운로드
from utils.download_lol import download_lol_dataset

try:
    download_lol_dataset(LOL_DIR)
except RuntimeError as e:
    print(f'자동 다운로드 실패: {e}')
    print('LOL 없이 커스텀 데이터만으로 진행합니다.')

In [ ]:
# Google Drive에서 커스텀 데이터 복사
# 커스텀 데이터 구조:
#   DRIVE_CUSTOM_DATA/
#   ├── train_input_img/   (빛 번짐 있는 원본)
#   ├── train_label_img/   (개선된 참고 이미지)
#   └── test_input_img/    (테스트 이미지)

import shutil

if os.path.exists(DRIVE_CUSTOM_DATA):
    shutil.copytree(DRIVE_CUSTOM_DATA, CUSTOM_DIR, dirs_exist_ok=True)
    # 복사된 파일 수 확인
    for folder in ['train_input_img', 'train_label_img', 'test_input_img']:
        path = os.path.join(CUSTOM_DIR, folder)
        if os.path.exists(path):
            count = len([f for f in os.listdir(path) if f.endswith('.png')])
            print(f'  {folder}: {count}장')
else:
    print(f'커스텀 데이터 없음: {DRIVE_CUSTOM_DATA}')
    print('LOL 데이터만으로 진행합니다.')

In [ ]:
# 통일 구조로 변환
from utils.prepare_data import prepare_data

meta = prepare_data(
    lol_dir=LOL_DIR,
    custom_dir=CUSTOM_DIR,
    output_dir=DATA_DIR,
    force=False,
)

print('\n데이터 준비 완료!')
print(f"  학습: {meta['total_train']}쌍")
print(f"  검증: {meta['total_val']}쌍")
print(f"  테스트: {meta['total_test']}장")

In [ ]:
# 데이터셋 샘플 확인
import matplotlib.pyplot as plt
from utils.data_loader import NightVisionDataset
from utils.augmentation import get_transform

train_ds = NightVisionDataset(DATA_DIR, split='train',
                               transform=get_transform('train'))
val_ds   = NightVisionDataset(DATA_DIR, split='val',
                               transform=get_transform('val'))
test_ds  = NightVisionDataset(DATA_DIR, split='test',
                               transform=get_transform('test'))

print(f'train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}')

# 샘플 시각화
sample = train_ds[0]
inp_np = sample['input'].permute(1, 2, 0).numpy()
tgt_np = sample['target'].permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(inp_np); axes[0].set_title('Input (저조도/빛 번짐)')
axes[1].imshow(tgt_np); axes[1].set_title('Target (개선된 이미지)')
for ax in axes: ax.axis('off')
plt.suptitle(f"샘플: {sample['filename']}")
plt.tight_layout()
plt.show()

---
## 5. Stage 1: LOL 사전학습

> ⚠️ **STEP 5 (training/train.py) 구현 후 활성화**

In [ ]:
# TODO: STEP 3,4,5 구현 후 아래 코드 활성화

# PRETRAIN_CONFIG = {
#     'stage': 'pretrain',
#     'data_dir': DATA_DIR,
#     'epochs': 100,
#     'batch_size': 16,
#     'lr': 1e-4,
#     'input_size': 192,
#     'device': 'cuda',
#     'save_dir': 'models/pretrained',
#     'save_path': 'models/pretrained/zerodce_pretrain.pt',
# }
#
# from training.train import Trainer
# trainer = Trainer(**PRETRAIN_CONFIG)
# trainer.train()

print('[TODO] STEP 5 구현 후 활성화됩니다.')

---
## 6. Stage 2: 커스텀 데이터 파인튜닝

> ⚠️ **STEP 5 (training/train.py) 구현 후 활성화**

In [ ]:
# TODO: STEP 3,4,5 구현 후 아래 코드 활성화

# FINETUNE_CONFIG = {
#     'stage': 'finetune',
#     'data_dir': DATA_DIR,
#     'epochs': 50,
#     'batch_size': 8,
#     'lr': 1e-5,
#     'input_size': 192,
#     'device': 'cuda',
#     'load_path': 'models/pretrained/zerodce_pretrain.pt',
#     'save_path': 'models/pretrained/zerodce_finetuned.pt',
# }
#
# from training.train import Trainer
# trainer = Trainer(**FINETUNE_CONFIG)
# trainer.train()

print('[TODO] STEP 5 구현 후 활성화됩니다.')

---
## 7. 결과 시각화

> 학습 완료 후 실행

In [ ]:
# TODO: STEP 5 이후 활성화
# from utils.visualization import show_results
# show_results(model, test_ds, num_samples=5)

print('[TODO] STEP 5 구현 후 활성화됩니다.')

---
## 8. 모델 Google Drive에 저장

In [ ]:
# 학습된 모델을 Drive에 저장
import shutil
import glob

model_files = glob.glob('models/pretrained/*.pt') + glob.glob('models/pretrained/*.onnx')

if model_files:
    for f in model_files:
        dst = os.path.join(DRIVE_MODEL_SAVE, os.path.basename(f))
        shutil.copy2(f, dst)
        print(f'저장됨: {dst}')
else:
    print('저장할 모델 파일이 없습니다. (학습 후 실행하세요)')